# Test inference

Attach the soil competition data and the soil-cv-utils code dataset for your models.
Enable Internet and the WANDB_API_KEY secret. A GPU makes inference faster.
Set CHECKPOINTS below, then Save & Run All.
CSV files are saved in /kaggle/working/submissions.

In [1]:
from pathlib import Path

CHECKPOINTS = [
    {
        "name": "swin-full-4fold",
        "checkpoints": [
            "public-team/soil-grain-size/swin-full-fold-0:v0",
            "public-team/soil-grain-size/swin-full-fold-1:v0",
            "public-team/soil-grain-size/swin-full-fold-2:v0",
            "public-team/soil-grain-size/swin-full-fold-3:v0"
        ]
    },
    {
        "name": "swin-last-stage-4fold",
        "checkpoints": [
            "public-team/soil-grain-size/swin-last-stage-fold-0:v0",
            "public-team/soil-grain-size/swin-last-stage-fold-1:v0",
            "public-team/soil-grain-size/swin-last-stage-fold-2:v0",
            "public-team/soil-grain-size/swin-last-stage-fold-3:v0"
        ]
    },
    {
        "name": "swin-frozen-4fold",
        "checkpoints": [
            "public-team/soil-grain-size/swin-frozen-fold-0:v0",
            "public-team/soil-grain-size/swin-frozen-fold-1:v0",
            "public-team/soil-grain-size/swin-frozen-fold-2:v0",
            "public-team/soil-grain-size/swin-frozen-fold-3:v0"
        ]
    },
    {
        "name": "vit-scratch-no-warmup-4fold",
        "checkpoints": [
            "public-team/soil-grain-size/vit-scratch-no-warmup-fold-0:v0",
            "public-team/soil-grain-size/vit-scratch-no-warmup-fold-1:v0",
            "public-team/soil-grain-size/vit-scratch-no-warmup-fold-2:v0",
            "public-team/soil-grain-size/vit-scratch-no-warmup-fold-3:v0"
        ]
    },
    {
        "name": "vit-scratch-p16-4fold",
        "checkpoints": [
            "public-team/soil-grain-size/vit-scratch-p16-fold-0:v0",
            "public-team/soil-grain-size/vit-scratch-p16-fold-1:v0",
            "public-team/soil-grain-size/vit-scratch-p16-fold-2:v0",
            "public-team/soil-grain-size/vit-scratch-p16-fold-3:v0"
        ]
    }
]
DEVICE = "auto"
OUTPUT_DIR = Path("/kaggle/working/submissions")


## Load the code

Use a run ID, a run URL, or an exact W&B artifact for one model.
Use a named group of four checkpoints for a four-fold ensemble.
Use None for an empty slot.
The package checks model settings, source compatibility and fold IDs.

In [2]:
import os
import sys

import torch
import wandb
from kaggle_secrets import UserSecretsClient

code_root = Path(os.environ.get(
    "SOILCV_INFERENCE_CODE_ROOT",
    "/kaggle/input/datasets/denysmykhailov/soil-cv-utils",
))
if not (code_root / "soilcv/inference/runner.py").is_file():
    matches = list(Path("/kaggle/input").rglob("soilcv/inference/runner.py"))
    if len(matches) != 1:
        raise RuntimeError("Attach the soil-cv-utils code dataset.")
    code_root = matches[0].parents[2]
sys.path.insert(0, str(code_root))
from soilcv.artifacts import package_identity
from soilcv.inference.runner import run_checkpoints


try:
    key = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception as error:
    raise RuntimeError("Enable WANDB_API_KEY in Add-ons > Secrets, then Run All.") from error
os.environ["WANDB_API_KEY"] = key
wandb.login(key=key, verify=True)
del key
print("GPU available:", torch.cuda.is_available())
print("Code:", package_identity())


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: dmykhailov (dmykhailov-kyiv-school-of-economics) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


GPU available: False
Code: {'version': '0.1.1', 'source_hash': '788545b08a93b55e34e5c0ce32a7c4fb141646aeed5cab3fc0f4740f7bbb7b44', 'folds_hash': '057fe8f93ba35d2ab7a3483063c95ca0079c4bcbb9978671c16ffc4119d9c1b2'}


## Predict the test set

Weights come from W&B. Each model uses its saved input settings.
A four-fold group averages the curves for each test sample.
Rows follow sample_submission.csv. IDs and valid curves are checked.
A group is skipped if a checkpoint is not ready.

In [3]:
results = run_checkpoints(CHECKPOINTS, OUTPUT_DIR, device=DEVICE)
print("Saved CSV files:", len(results))

[1] Resolve 4 checkpoint(s)


wandb: Downloading large artifact 'swin-full-fold-0:v0', 105.30MB. 2 files...
wandb:   2 of 2 files downloaded.  
Done. 00:00:08.5 (12.4MB/s)
wandb: Downloading large artifact 'swin-full-fold-1:v0', 105.30MB. 2 files...
wandb:   2 of 2 files downloaded.  
Done. 00:00:07.8 (13.4MB/s)
wandb: Downloading large artifact 'swin-full-fold-2:v0', 105.30MB. 2 files...
wandb:   2 of 2 files downloaded.  
Done. 00:00:08.8 (12.0MB/s)
wandb: Downloading large artifact 'swin-full-fold-3:v0', 105.30MB. 2 files...
wandb:   2 of 2 files downloaded.  
Done. 00:00:08.8 (12.0MB/s)


Saved: submission_01_swin-full-4fold.csv (10, 12)
[2] Resolve 4 checkpoint(s)


wandb: Downloading large artifact 'swin-last-stage-fold-0:v0', 105.30MB. 2 files...
wandb:   2 of 2 files downloaded.  
Done. 00:00:06.9 (15.4MB/s)
wandb: Downloading large artifact 'swin-last-stage-fold-1:v0', 105.30MB. 2 files...
wandb:   2 of 2 files downloaded.  
Done. 00:00:08.4 (12.5MB/s)
wandb: Downloading large artifact 'swin-last-stage-fold-2:v0', 105.30MB. 2 files...
wandb:   2 of 2 files downloaded.  
Done. 00:00:08.8 (11.9MB/s)
wandb: Downloading large artifact 'swin-last-stage-fold-3:v0', 105.30MB. 2 files...
wandb:   2 of 2 files downloaded.  
Done. 00:00:07.1 (14.9MB/s)


Saved: submission_02_swin-last-stage-4fold.csv (10, 12)
[3] Resolve 4 checkpoint(s)


...

Saved: submission_03_swin-frozen-4fold.csv (10, 12)
[4] Resolve 4 checkpoint(s)


wandb:   2 of 2 files downloaded.  
wandb:   2 of 2 files downloaded.  
wandb:   2 of 2 files downloaded.  
wandb:   2 of 2 files downloaded.  


Saved: submission_04_vit-scratch-no-warmup-4fold.csv (10, 12)
[5] Resolve 4 checkpoint(s)


wandb:   2 of 2 files downloaded.  
wandb:   2 of 2 files downloaded.  
wandb:   2 of 2 files downloaded.  
wandb:   2 of 2 files downloaded.  


Saved: submission_05_vit-scratch-p16-4fold.csv (10, 12)
Saved CSV files: 5


## Ready submissions

One CSV belongs to one model or one complete four-fold ensemble.
inference_manifest.json records the model versions and source runs.
Files appear in Kaggle Outputs after Save & Run All ends.
The notebook does not train models or submit files to the competition.

In [4]:
import pandas as pd
from IPython.display import FileLink, display

assert results, "No ready predictions yet."
for result in results:
    path = Path(result["output"])
    frame = pd.read_csv(path)
    print(path.name, "| shape:", frame.shape)
    display(FileLink(str(path)))
    display(frame.head(2))
display(pd.DataFrame(results)[["slot", "name", "ensemble", "device", "output"]])
print("Outputs:", OUTPUT_DIR)


submission_01_swin-full-4fold.csv | shape: (10, 12)


/kaggle/working/submissions/submission_01_swin-full-4fold.csv

,sample_id,0.002,0.0063,0.02,0.063,0.2,0.63,2,6.3,20,63,200
0,HPC_Airbus BS6-3,2.683370,5.589636,15.321021,40.34837,73.689836,86.537586,90.353809,94.939772,98.575806,99.925135,100.0
1,HPC_Audorfring,1.761432,3.648441,9.356734,23.75995,45.993926,63.350299,72.321170,83.580839,94.777166,99.882784,100.0


submission_02_swin-last-stage-4fold.csv | shape: (10, 12)


/kaggle/working/submissions/submission_02_swin-last-stage-4fold.csv

...

submission_05_vit-scratch-p16-4fold.csv | shape: (10, 12)


/kaggle/working/submissions/submission_05_vit-scratch-p16-4fold.csv

,sample_id,0.002,0.0063,0.02,0.063,0.2,0.63,2,6.3,20,63,200
0,HPC_Airbus BS6-3,2.070254,4.321421,11.207935,27.784112,46.648584,57.349327,67.278362,82.559616,95.118324,99.709490,100.0
1,HPC_Audorfring,2.087338,4.355877,11.266078,27.842452,46.561636,57.114949,67.071136,82.453188,95.094725,99.707928,100.0


,slot,name,ensemble,device,output
0,1,swin-full-4fold,True,cpu,/kaggle/working/submissions/submission_01_swin...
1,2,swin-last-stage-4fold,True,cpu,/kaggle/working/submissions/submission_02_swin...
2,3,swin-frozen-4fold,True,cpu,/kaggle/working/submissions/submission_03_swin...
3,4,vit-scratch-no-warmup-4fold,True,cpu,/kaggle/working/submissions/submission_04_vit-...
4,5,vit-scratch-p16-4fold,True,cpu,/kaggle/working/submissions/submission_05_vit-...


Outputs: /kaggle/working/submissions
